In [ ]:
import os
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")

In [ ]:
loader = TextLoader("documents/real_madrid_rag_dataset.txt", encoding="utf-8")
docs = loader.load()

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
splits = text_splitter.split_documents(docs)

In [ ]:
splits

In [ ]:
!pip install -U langchain-google-genai

In [ ]:
!pip install -U langchain_huggingface

In [ ]:
!pip install -U langchain_chroma

In [ ]:
import os

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_chroma import Chroma

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    directory="chroma_db"
)

In [ ]:
pip install sentence-transformers

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

In [ ]:
pip install -U langchain-chroma chromadb

In [ ]:
from langchain_chroma import Chroma

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader(
    "documents/real_madrid_rag_dataset.txt",
    encoding="utf-8"
)

docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

splits = text_splitter.split_documents(docs)

print("Количество частей:", len(splits))

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print("Embeddings created")

In [ ]:
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory="chroma_db"
)

In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are an AI assistant specialized in Real Madrid.

Answer the user's question using ONLY the provided context.
If the answer is not found, say:
"I couldn't find that information in my knowledge base."

Context:
{context}

Question:
{question}

Answer:
""")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os

api_key = os.getenv("GOOGLE_API_KEY")

llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite-preview",
    temperature=0
)

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
query = "When was Real Madrid founded?"

response = rag_chain.invoke(query)

print(response)

In [ ]:
docs = retriever.invoke("Real Madrid")

print(len(docs))

for doc in docs:
    print(doc.page_content[:300])
    print("-" * 50)

In [ ]:
docs = retriever.invoke("When was Real Madrid founded?")

for i, doc in enumerate(docs, 1):
    print(f"DOCUMENT {i}")
    print(doc.page_content[:500])
    print("-" * 70)

In [ ]:
results = vectorstore.similarity_search(
    "When was Real Madrid founded?",
    k=10
)

for i, doc in enumerate(results, 1):
    print(f"\nDOCUMENT {i}")
    print(doc.page_content[:700])
    print("-" * 70)

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 8,
        "fetch_k": 20
    }
)

In [ ]:
loader = TextLoader(
    "documents/real_madrid_rag_dataset.txt",
    encoding="utf-8"
)

source_docs = loader.load()

splits = text_splitter.split_documents(source_docs)

print("Chunks:", len(splits))
print("1902 exists:", any("1902" in chunk.page_content for chunk in splits))

In [ ]:
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory="chroma_db_v2"
)

In [ ]:
results = vectorstore.similarity_search(
    "When was Real Madrid founded?",
    k=10
)

for i, doc in enumerate(results, 1):
    print(f"DOCUMENT {i}")
    print(doc.page_content)
    print("-" * 70)

In [ ]:
results = vectorstore.similarity_search(
    "Real Madrid foundation 1902 founded history",
    k=34
)

for i, doc in enumerate(results, 1):
    if "1902" in doc.page_content:
        print("FOUND IN DOCUMENT:", i)
        print(doc.page_content)
        break

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 6}
)

In [ ]:
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
response = rag_chain.invoke("When was Real Madrid founded?")
print(response)

In [ ]:
response = rag_chain.invoke("Who is the all-time top scorer of Real Madrid?")
print(response)